## Setup


In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer, normalize

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
OUTPUTS_TASK2_DIR = PROJECT_ROOT / "outputs" / "task2"
DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = DATA_DIR / "cache"

TFIDF_DEFAULTS = {
    "max_features": 5000,
    "min_df": 2,
    "sublinear_tf": True,
}

RANDOM_STATE = 42  # for reproducibility. set to None for random

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

## Preprocessing


In [8]:
def _clean_text(text: object) -> str:
    if not isinstance(text, str):
        return ""

    t = text.lower()

    # Remove LaTeX math blocks and commands.
    t = re.sub(r"\$.*?\$|\\\[.*?\\\]|\\\(.*?\\\)", " ", t, flags=re.DOTALL)
    t = re.sub(
        r"\\(?:begin|end)\{[^{}]*\}|\\[a-zA-Z]+\*?(?:\[[^\]]*\])?(?:\{[^{}]*\})*",
        " ",
        t,
    )

    # Remove URLs and numeric citation brackets.
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"\[[0-9,\s]+\]", " ", t)

    # Keep only alphanumeric, hyphen, underscore, and spaces.
    t = re.sub(r"[^a-z0-9\-_ ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def load_train_test_cleaned_data_local(
    project_root: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_cache = project_root / "data" / "cache" / "train_cleaned.csv"
    test_cache = project_root / "data" / "cache" / "test_cleaned.csv"

    if train_cache.exists() and test_cache.exists():
        train_df = pd.read_csv(train_cache, sep="	")
        test_df = pd.read_csv(test_cache, sep="	")
        train_df = train_df[["id", "label_id", "cleaned_abstract"]].copy()
        test_df = test_df[["id", "cleaned_abstract"]].copy()
        return train_df, test_df

    train_path = project_root / "data" / "train.csv"
    test_path = project_root / "data" / "test.csv"
    train_raw = pd.read_csv(train_path, sep="\t")
    test_raw = pd.read_csv(test_path, sep="\t")

    train_df = train_raw[["id", "label_id"]].copy()
    test_df = test_raw[["id"]].copy()
    train_df["cleaned_abstract"] = train_raw["abstract"].map(_clean_text)
    test_df["cleaned_abstract"] = test_raw["abstract"].map(_clean_text)
    return train_df, test_df


train_df, test_df = load_train_test_cleaned_data_local(PROJECT_ROOT)
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Classes: {train_df['label_id'].nunique()}")

Train shape: (139156, 3)
Test shape: (34790, 2)
Classes: 39


## Task 1: Custom Logistic Regression


### Task 1 Hyperparameters


In [9]:
EPOCHS = 10
BATCH_SIZE = 512
LEARNING_RATE = 2.0
REG_STRENGTH = 1e-4

### Binary Logistic Regression


In [10]:
class LogisticRegression:
    def __init__(self):
        self.weights = None
        self.bias = None

    @staticmethod
    def sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    def gradients(self, X, y, y_hat, reg_strength, sample_weight):
        weight_sum = float(np.sum(sample_weight))
        error = (y_hat - y) * sample_weight
        dw = (1.0 / weight_sum) * (X.T @ error) + reg_strength * self.weights
        db = (1.0 / weight_sum) * np.sum(error)
        return dw, db

    def train(self, X, y, bs, epochs, lr, reg_strength=1e-4):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features, dtype=np.float64)
        self.bias = 0.0

        y_arr = np.asarray(y)
        pos_count = int(np.sum(y_arr == 1))
        neg_count = n_samples - pos_count

        pos_w = n_samples / (2.0 * pos_count)
        neg_w = n_samples / (2.0 * neg_count)

        rng = np.random.default_rng(RANDOM_STATE)
        for _ in range(epochs):
            order = rng.permutation(n_samples)
            X_epoch = X[order]
            y_epoch = y_arr[order]

            for i in range(0, n_samples, bs):
                X_batch = X_epoch[i : i + bs]
                y_batch = y_epoch[i : i + bs]
                sample_weight = np.where(y_batch == 1, pos_w, neg_w).astype(np.float64)

                y_hat = self.sigmoid(X_batch @ self.weights + self.bias)
                dw, db = self.gradients(
                    X_batch, y_batch, y_hat, reg_strength, sample_weight
                )
                self.weights -= lr * dw
                self.bias -= lr * db

### OvR Multiclass Adaptation


In [11]:
class MultiClassLogisticRegression:
    def __init__(self):
        self.models = []
        self.n_classes = None

    def train(self, X, y, bs=32, epochs=100, lr=0.01, reg_strength=1e-4):
        unique_classes = np.unique(y)
        self.n_classes = len(unique_classes)
        self.models = []

        for c in range(self.n_classes):
            y_binary = (y == c).astype(int)
            model = LogisticRegression()
            model.train(
                X, y_binary, bs=bs, epochs=epochs, lr=lr, reg_strength=reg_strength
            )
            self.models.append(model)

    def predict(self, X):
        if self.n_classes is None:
            raise ValueError("Model is not trained yet.")

        n_samples = int(X.shape[0])
        n_classes = int(self.n_classes)
        probs = np.zeros((n_samples, n_classes), dtype=np.float64)
        for c, model in enumerate(self.models):
            linear_output = X @ model.weights + model.bias
            probs[:, c] = model.sigmoid(linear_output)
        return np.argmax(probs, axis=1)

### sklearn Logistic Regression Comparison


In [12]:
# Prepare data for task 1
X_text = train_df["cleaned_abstract"].to_numpy(dtype=str)
y = train_df["label_id"].to_numpy(dtype=np.int64)

X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer_task1 = TfidfVectorizer(**TFIDF_DEFAULTS)
X_train = vectorizer_task1.fit_transform(X_train_text)
X_val = vectorizer_task1.transform(X_val_text)

In [13]:
# Predict with custom OvR logreg
custom_model = MultiClassLogisticRegression()
custom_model.train(
    X_train,
    y_train,
    bs=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    reg_strength=REG_STRENGTH,
)
custom_pred = custom_model.predict(X_val).astype(np.int64)

In [14]:
# Predict with sklearn OvR logreg for comparison
base_lr = SkLogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
)
sk_model = OneVsRestClassifier(base_lr)
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_val).astype(np.int64)

In [15]:
comparison_df = pd.DataFrame(
    [
        {
            "model": "custom_logreg_ovr",
            "macro_f1": f1_score(y_val, custom_pred, average="macro"),
            "accuracy": accuracy_score(y_val, custom_pred),
        },
        {
            "model": "sklearn_logreg_ovr",
            "macro_f1": f1_score(y_val, sk_pred, average="macro"),
            "accuracy": accuracy_score(y_val, sk_pred),
        },
    ]
).sort_values(by=["macro_f1", "accuracy"], ascending=False)

comparison_df

,model,macro_f1,accuracy
1,sklearn_logreg_ovr,0.634325,0.723556
0,custom_logreg_ovr,0.616918,0.693123


### Task 1 Test Prediction Export (`LogReg_Prediction.csv`)


In [ ]:
vectorizer_full = TfidfVectorizer(**TFIDF_DEFAULTS)
X_full_train = vectorizer_full.fit_transform(
    train_df["cleaned_abstract"].to_numpy(dtype=str)
)
X_full_test = vectorizer_full.transform(test_df["cleaned_abstract"].to_numpy(dtype=str))
y_full_train = train_df["label_id"].to_numpy(dtype=np.int64)

final_custom_model = MultiClassLogisticRegression()
final_custom_model.train(
    X_full_train,
    y_full_train,
    bs=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    reg_strength=REG_STRENGTH,
)
final_test_pred = final_custom_model.predict(X_full_test).astype(np.int64)

logreg_submission = pd.DataFrame(
    {
        "id": test_df["id"].astype(np.int64),
        "label_id": final_test_pred,
    }
)

if list(logreg_submission.columns) != ["id", "label_id"]:
    raise ValueError("Task 1 submission columns must be exactly: id,label_id")
if len(logreg_submission) != len(test_df):
    raise ValueError("Task 1 submission row count mismatch.")

logreg_output_path = SUBMISSIONS_DIR / "LogReg_Prediction.csv"
logreg_submission.to_csv(logreg_output_path, index=False)
print(f"Saved: {logreg_output_path}")

Saved: g:\Projects\Dev\sutd\ml\ml-project\submissions\LogReg_Prediction.csv
       id  label_id
0  173148        38
1   29098         4
2   28211         4
3  136101         0
4   97133        21


## Task 2: KNN with Feature Selection and Dimension Reduction


### Task 2 Hyperparameters


In [ ]:
TASK2_SIZES = [2000, 1000, 500, 100]

N_SPLITS = 5

# KNN settings for Task 2
N_NEIGHBORS = 2
KNN_WEIGHTS = "distance"
KNN_METRIC = "cosine"
KNN_ALGORITHM = "brute"
KNN_N_JOBS = 4

# If false reads from output folder, else recompute results (which are saved to output folder)
RECOMPUTE_TASK2_RESULTS = True

RUN_TASK2_SUBMISSION_GENERATION = False


def build_knn(n_jobs: int = KNN_N_JOBS) -> KNeighborsClassifier:
    return KNeighborsClassifier(
        n_neighbors=N_NEIGHBORS,
        weights=KNN_WEIGHTS,
        metric=KNN_METRIC,
        algorithm=KNN_ALGORITHM,
        n_jobs=n_jobs,
    )

### Task 2 Validation Experiments (CV)


#### Feature Selection (Top-N TF-IDF)


In [21]:
def run_feature_selection_cv_knn(train_df: pd.DataFrame) -> pd.DataFrame:
    texts = train_df["cleaned_abstract"].tolist()
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_indices = list(skf.split(texts, y))

    rows = []
    for max_features in TASK2_SIZES:
        fold_f1 = []
        fold_acc = []

        for train_idx, val_idx in fold_indices:
            train_texts = [texts[i] for i in train_idx]
            val_texts = [texts[i] for i in val_idx]
            y_train = y[train_idx]
            y_val = y[val_idx]

            vec = TfidfVectorizer(**{**TFIDF_DEFAULTS, "max_features": max_features})
            X_train = vec.fit_transform(train_texts)
            X_val = vec.transform(val_texts)

            clf = build_knn(n_jobs=KNN_N_JOBS)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_val).astype(np.int64)

            fold_f1.append(f1_score(y_val, y_pred, average="macro"))
            fold_acc.append(accuracy_score(y_val, y_pred))

        rows.append(
            {
                "method": "feature_selection",
                "max_features": int(max_features),
                "mean_macro_f1": float(np.mean(fold_f1)),
                "std_macro_f1": float(np.std(fold_f1)),
                "mean_accuracy": float(np.mean(fold_acc)),
                "std_accuracy": float(np.std(fold_acc)),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(by="max_features", ascending=False)
        .reset_index(drop=True)
    )

#### Dimension Reduction (TruncatedSVD)


In [22]:
def _build_dim_reduction_pipeline(n_components: int) -> Pipeline:
    return Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(**TFIDF_DEFAULTS)),
            (
                "svd",
                TruncatedSVD(
                    n_components=n_components, n_iter=7, random_state=RANDOM_STATE
                ),
            ),
            ("normalize", Normalizer(norm="l2")),
            (
                "knn",
                build_knn(n_jobs=1),
            ),
        ]
    )


def run_dimension_reduction_cv_knn(train_df: pd.DataFrame) -> pd.DataFrame:
    X = train_df["cleaned_abstract"].to_numpy(dtype=str)
    y = train_df["label_id"].to_numpy(dtype=np.int64)

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {"macro_f1": "f1_macro", "accuracy": "accuracy"}

    rows = []
    for n_components in TASK2_SIZES:
        pipe = _build_dim_reduction_pipeline(n_components)
        cv_result = cross_validate(
            estimator=pipe,
            X=X,
            y=y,
            cv=cv,
            scoring=scoring,
            n_jobs=KNN_N_JOBS,
            return_train_score=False,
            error_score="raise",
        )

        rows.append(
            {
                "method": "dimension_reduction",
                "n_components": int(n_components),
                "mean_macro_f1": float(np.mean(cv_result["test_macro_f1"])),
                "std_macro_f1": float(np.std(cv_result["test_macro_f1"])),
                "mean_accuracy": float(np.mean(cv_result["test_accuracy"])),
                "std_accuracy": float(np.std(cv_result["test_accuracy"])),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(by="n_components", ascending=False)
        .reset_index(drop=True)
    )

#### CV Result Reporting (Load Existing CSVs)


In [23]:
fs_cv_path = OUTPUTS_TASK2_DIR / "feature_selection_knn_cv_results.csv"
svd_cv_path = OUTPUTS_TASK2_DIR / "dimension_reduction_knn_cv_results_sklearn.csv"

if RECOMPUTE_TASK2_RESULTS:
    OUTPUTS_TASK2_DIR.mkdir(parents=True, exist_ok=True)
    run_feature_selection_cv_knn(train_df).to_csv(fs_cv_path, index=False)
    run_dimension_reduction_cv_knn(train_df).to_csv(svd_cv_path, index=False)

fs_raw = pd.read_csv(fs_cv_path)
svd_raw = pd.read_csv(svd_cv_path)

fs_results = fs_raw[
    [
        "max_features",
        "mean_macro_f1",
        "std_macro_f1",
        "mean_accuracy",
        "std_accuracy",
    ]
].copy()
fs_results.insert(0, "method", "feature_selection")

svd_results = svd_raw[
    [
        "n_components",
        "mean_macro_f1",
        "std_macro_f1",
        "mean_accuracy",
        "std_accuracy",
    ]
].copy()
svd_results.insert(0, "method", "dimension_reduction")

# Combined compact report table
fs_compact = fs_results.rename(columns={"max_features": "size"}).copy()
fs_compact["method"] = "FS"
svd_compact = svd_results.rename(columns={"n_components": "size"}).copy()
svd_compact["method"] = "SVD"

combined_task2_report = pd.concat([fs_compact, svd_compact], ignore_index=True)
combined_task2_report = combined_task2_report[
    ["method", "size", "mean_macro_f1", "std_macro_f1", "mean_accuracy", "std_accuracy"]
]
combined_task2_report = combined_task2_report.sort_values(
    by=["method", "size"], ascending=[True, False]
).reset_index(drop=True)

print("Combined Task 2 report table:")
display(combined_task2_report)

Combined Task 2 report table:


,method,size,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy
0,FS,2000,0.529799,0.004757,0.620677,0.002527
1,FS,1000,0.483986,0.002018,0.581707,0.001167
2,FS,500,0.421616,0.002511,0.526991,0.001993
3,FS,100,0.229744,0.002120,0.312606,0.002710
4,SVD,2000,0.556718,0.003655,0.644593,0.002106
5,SVD,1000,0.550087,0.003417,0.641719,0.001290
6,SVD,500,0.539813,0.002111,0.637515,0.000545
7,SVD,100,0.510225,0.003399,0.618234,0.002293


### Task 2 Test Submission Generation


In [24]:
def generate_task2_feature_selection_submissions(
    train_df: pd.DataFrame, test_df: pd.DataFrame
) -> list[Path]:
    train_texts = train_df["cleaned_abstract"].tolist()
    test_texts = test_df["cleaned_abstract"].tolist()
    y_train = train_df["label_id"].to_numpy(dtype=np.int64)

    output_paths = []
    for max_features in TASK2_SIZES:
        vec = TfidfVectorizer(**{**TFIDF_DEFAULTS, "max_features": max_features})
        X_train = vec.fit_transform(train_texts)
        X_test = vec.transform(test_texts)

        clf = build_knn(n_jobs=KNN_N_JOBS)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test).astype(np.int64)

        out_path = SUBMISSIONS_DIR / f"task2_knn_fs_{max_features}.csv"
        sub_df = pd.DataFrame(
            {"id": test_df["id"].astype(np.int64), "label_id": y_pred}
        )
        if list(sub_df.columns) != ["id", "label_id"]:
            raise ValueError("Submission columns must be exactly: id,label_id")
        if len(sub_df) != len(test_df):
            raise ValueError("Submission row count mismatch.")
        sub_df.to_csv(out_path, index=False)
        output_paths.append(out_path)

    return output_paths


def generate_task2_svd_submissions(
    train_df: pd.DataFrame, test_df: pd.DataFrame
) -> list[Path]:
    train_texts = train_df["cleaned_abstract"].tolist()
    test_texts = test_df["cleaned_abstract"].tolist()
    y_train = train_df["label_id"].to_numpy(dtype=np.int64)

    vec = TfidfVectorizer(**TFIDF_DEFAULTS)
    X_train = vec.fit_transform(train_texts)
    X_test = vec.transform(test_texts)

    output_paths = []
    for n_components in TASK2_SIZES:
        svd = TruncatedSVD(n_components=n_components, n_iter=7)
        X_train_reduced = svd.fit_transform(X_train).astype(np.float32, copy=False)
        X_test_reduced = svd.transform(X_test).astype(np.float32, copy=False)

        normalize(X_train_reduced, norm="l2", copy=False)
        normalize(X_test_reduced, norm="l2", copy=False)

        clf = build_knn(n_jobs=KNN_N_JOBS)
        clf.fit(X_train_reduced, y_train)
        y_pred = clf.predict(X_test_reduced).astype(np.int64)

        out_path = SUBMISSIONS_DIR / f"task2_knn_svd_{n_components}.csv"
        sub_df = pd.DataFrame(
            {"id": test_df["id"].astype(np.int64), "label_id": y_pred}
        )
        if list(sub_df.columns) != ["id", "label_id"]:
            raise ValueError("Submission columns must be exactly: id,label_id")
        if len(sub_df) != len(test_df):
            raise ValueError("Submission row count mismatch.")
        sub_df.to_csv(out_path, index=False)
        output_paths.append(out_path)

    return output_paths


if RUN_TASK2_SUBMISSION_GENERATION:
    fs_paths = generate_task2_feature_selection_submissions(train_df, test_df)
    svd_paths = generate_task2_svd_submissions(train_df, test_df)
    print("Generated FS submissions:")
    for p in fs_paths:
        print(" -", p)
    print("Generated SVD submissions:")
    for p in svd_paths:
        print(" -", p)

### Kaggle Test Macro F1

| Task                               | Submission File          | Score |
| ---------------------------------- | ------------------------ | ----: |
| Task 1                             | `LogReg_Prediction.csv`  | 0.620 |
| Task 2 (Feature Selection, 2000)   | `task2_knn_fs_2000.csv`  | 0.549 |
| Task 2 (Feature Selection, 1000)   | `task2_knn_fs_1000.csv`  | 0.496 |
| Task 2 (Feature Selection, 500)    | `task2_knn_fs_500.csv`   | 0.436 |
| Task 2 (Feature Selection, 100)    | `task2_knn_fs_100.csv`   | 0.238 |
| Task 2 (Dimension Reduction, 2000) | `task2_knn_svd_2000.csv` | 0.581 |
| Task 2 (Dimension Reduction, 1000) | `task2_knn_svd_1000.csv` | 0.567 |
| Task 2 (Dimension Reduction, 500)  | `task2_knn_svd_500.csv`  | 0.562 |
| Task 2 (Dimension Reduction, 100)  | `task2_knn_svd_100.csv`  | 0.522 |
| Task 3                             | `task3_linear_svc.csv`   | 0.677 |
